# Autogration Run Inspection

Fetch and inspect one completed RESOLVE simulation using the live Autogration API when available, otherwise the checked-in exported-run fixture. This proof is read-only and uses no fabricated data.

**Provenance:** the fallback is `autogration/tests/backend/datafiles/run_result_unpacked.json`, produced from a real test simulation result and shaped like Autogration's CDF time-series export. A fixture result demonstrates parsing and visualization, not live CDF connectivity.

## Acquire one exported run

The cell below first asks the local read-only Autogration API for a completed PETEX run. If the API or database is unavailable, it loads the checked-in exported simulation fixture and records that fallback explicitly.

In [1]:
import json
from pathlib import Path
from urllib.error import URLError
from urllib.request import urlopen

API_BASE_URL = "http://localhost:5201/api/v1"
FIXTURE_CANDIDATES = (
    Path.cwd() / "autogration/tests/backend/datafiles/run_result_unpacked.json",
    Path.cwd() / "../autogration/tests/backend/datafiles/run_result_unpacked.json",
)


def read_json(url: str):
    with urlopen(url, timeout=2) as response:
        return json.load(response)


source_kind = "live_api"
source_detail = API_BASE_URL
selected_run = None
try:
    run_page = read_json(
        f"{API_BASE_URL}/runs/?status=Completed&type=petex_compute&limit=1"
    )
    completed_runs = run_page.get("runs", [])
    if not completed_runs:
        raise LookupError("The local API has no completed PETEX runs.")
    selected_run = completed_runs[0]
    exported_series = read_json(
        f"{API_BASE_URL}/runs/{selected_run['id']}/results"
    )
except (OSError, URLError, TimeoutError, LookupError) as error:
    fixture_path = next(path.resolve() for path in FIXTURE_CANDIDATES if path.exists())
    with fixture_path.open(encoding="utf-8") as fixture_file:
        exported_series = json.load(fixture_file)
    source_kind = "checked_in_export_fixture"
    source_detail = str(fixture_path)
    acquisition_note = str(error)
else:
    acquisition_note = "Fetched the latest completed PETEX run from the local API."

print(
    json.dumps(
        {
            "source_kind": source_kind,
            "source_detail": source_detail,
            "selected_run": selected_run,
            "top_level_records": len(exported_series),
            "note": acquisition_note,
        },
        indent=2,
        default=str,
    )
)

{
  "source_kind": "checked_in_export_fixture",
  "source_detail": "C:\\Users\\aleksekk\\autogration_v2\\autogration\\tests\\backend\\datafiles\\run_result_unpacked.json",
  "selected_run": null,
  "top_level_records": 18,
  "note": "<urlopen error timed out>"
}


### Acquisition result

This execution used the checked-in CDF-shaped export fixture because the local API timed out. It contains 18 exported time-series records from the RESOLVE test simulation. The following analysis remains useful for validating Autogration's result shape, but it is not evidence of current CDF contents or permissions.

## Normalize and validate the export

Each CDF series is converted to a consistent catalog record plus timestamp/value observations. The checks retain missing values, flag duplicate timestamps, and verify that series identity and units are present.

In [6]:
from collections import Counter
from datetime import datetime

series_catalog = []
observations = []
for record in exported_series:
    time_series = record["ts"]
    metadata = time_series.get("metadata", {})
    external_id = time_series["external_id"]
    catalog_entry = {
        "external_id": external_id,
        "task_id": metadata.get("task_id"),
        "asset": metadata.get("asset"),
        "ensemble": metadata.get("ensemble"),
        "realization": metadata.get("realization"),
        "system_type": metadata.get("system_type"),
        "object_type": metadata.get("object_type"),
        "object": metadata.get("object"),
        "label": metadata.get("label"),
        "parameter": metadata.get("parameter"),
        "unit": time_series.get("unit"),
        "source": metadata.get("source"),
        "resource_type": metadata.get("resource_type"),
    }
    series_catalog.append(catalog_entry)
    for point in record.get("data", {}).get("datapoints", []):
        observations.append(
            {
                **catalog_entry,
                "timestamp": datetime.fromisoformat(point["timestamp"]),
                "value": point.get("value"),
            }
        )

series_counts = Counter(row["external_id"] for row in observations)
timestamp_counts = Counter(
    (row["external_id"], row["timestamp"]) for row in observations
)
missing_values = sum(row["value"] is None for row in observations)
duplicate_points = sum(count - 1 for count in timestamp_counts.values() if count > 1)
missing_identity = sum(
    not row["external_id"] or not row["task_id"] or not row["parameter"]
    for row in series_catalog
)
missing_units = sum(not row["unit"] for row in series_catalog)

inventory = {
    "series": len(series_catalog),
    "datapoints": len(observations),
    "task_ids": sorted({row["task_id"] for row in series_catalog}),
    "realizations": sorted({row["realization"] for row in series_catalog}),
    "systems": sorted({row["system_type"] for row in series_catalog}),
    "objects": len({(row["object_type"], row["object"]) for row in series_catalog}),
    "parameters": sorted({row["parameter"] for row in series_catalog}),
    "date_range": [
        min(row["timestamp"] for row in observations).isoformat(),
        max(row["timestamp"] for row in observations).isoformat(),
    ],
    "quality": {
        "missing_values": missing_values,
        "duplicate_series_timestamps": duplicate_points,
        "series_missing_identity": missing_identity,
        "series_missing_units": missing_units,
    },
}
print(json.dumps(inventory, indent=2, default=str))

{
  "series": 18,
  "datapoints": 882,
  "task_ids": [
    "123456789"
  ],
  "realizations": [
    "1"
  ],
  "systems": [
    "PROD"
  ],
  "objects": 1,
  "parameters": [
    "AvgGasRate",
    "AvgLiqRate",
    "AvgOilRate",
    "AvgWatRate",
    "CumGas",
    "CumOil",
    "CumWat",
    "DPC",
    "GOR",
    "GasRate",
    "LiqRate",
    "ManPres",
    "ManTemp",
    "OilRate",
    "Pres",
    "Temp",
    "WCT",
    "WatRate"
  ],
  "date_range": [
    "2025-01-01T00:00:00",
    "2029-01-01T00:00:00"
  ],
  "quality": {
    "missing_values": 0,
    "duplicate_series_timestamps": 0,
    "series_missing_identity": 0,
    "series_missing_units": 0
  }
}


### Normalization result

The fixture represents task `123456789`, realization `1`, with 18 parameters and 882 monthly values from January 2025 through January 2029. All series belong to the `PROD` system and one equipment object. The checked export has no null values, duplicate series/timestamp pairs, missing identity fields, or missing units.

## Summarize series behavior

This table uses every point in each series. It reports first, last, minimum, maximum, and relative change when a nonzero starting value makes that comparison meaningful.

In [7]:
from collections import defaultdict
from html import escape
from IPython.display import HTML, display

points_by_series = defaultdict(list)
for row in observations:
    if row["value"] is not None:
        points_by_series[row["external_id"]].append(row)

series_summaries = []
for catalog_entry in series_catalog:
    points = sorted(
        points_by_series[catalog_entry["external_id"]],
        key=lambda row: row["timestamp"],
    )
    values = [row["value"] for row in points]
    first_value = values[0]
    last_value = values[-1]
    relative_change = (
        100 * (last_value - first_value) / abs(first_value)
        if first_value not in (None, 0)
        else None
    )
    series_summaries.append(
        {
            "parameter": catalog_entry["parameter"],
            "object": catalog_entry["object"],
            "unit": catalog_entry["unit"],
            "points": len(points),
            "first": first_value,
            "last": last_value,
            "minimum": min(values),
            "maximum": max(values),
            "change_percent": relative_change,
        }
    )

headers = ["parameter", "object", "unit", "points", "first", "last", "minimum", "maximum", "change_percent"]
rows_html = []
for summary in sorted(series_summaries, key=lambda row: row["parameter"]):
    cells = []
    for header in headers:
        value = summary[header]
        rendered = "n/a" if value is None else f"{value:,.3f}" if isinstance(value, float) else str(value)
        cells.append(f"<td>{escape(rendered)}</td>")
    rows_html.append(f"<tr>{''.join(cells)}</tr>")

display(
    HTML(
        "<div style='max-height:440px;overflow:auto'>"
        "<table style='border-collapse:collapse;width:100%;font:13px sans-serif'>"
        "<thead><tr>"
        + "".join(f"<th style='position:sticky;top:0;background:#f3f4f6;padding:7px;text-align:left'>{escape(header)}</th>" for header in headers)
        + "</tr></thead><tbody>"
        + "".join(rows_html)
        + "</tbody></table></div>"
    )
)

parameter,object,unit,points,first,last,minimum,maximum,change_percent
AvgGasRate,FieldSep,Sm3/day,49,0.000,"21,543.700",0.000,"1,447,560.000",n/a
AvgLiqRate,FieldSep,Sm3/day,49,"4,824.900","1,328.520","1,328.520","4,825.180",-72.465
AvgOilRate,FieldSep,Sm3/day,49,"4,824.900",122.230,122.230,"4,825.180",-97.467
AvgWatRate,FieldSep,Sm3/day,49,0.000,"1,206.290",0.000,"1,206.890",n/a
CumGas,FieldSep,MSm3,49,0.000,544.839,0.000,544.839,n/a
CumOil,FieldSep,MSm3,49,0.000,2.046,0.000,2.046,n/a
CumWat,FieldSep,MSm3,49,0.000,1.368,0.000,1.368,n/a
DPC,FieldSep,bar,49,0.000,0.000,0.000,0.000,n/a
GOR,FieldSep,Sm3/Sm3,49,0.000,176.247,0.000,464.287,n/a
GasRate,FieldSep,Sm3/day,49,0.000,"22,325.100",0.000,"1,500,060.000",n/a


### Series behavior

The result has a clear forecast trajectory. `OilRate` declines from 4,999.9 to 126.663 Sm3/day (-97.5%), while `WCT` rises from 0% to 90.799%. `LiqRate` declines 72.5%, and cumulative oil reaches 2.046 MSm3. Relative change is intentionally reported as unavailable for series beginning at zero.

## Explore the run interactively

Choose any exported parameter. The chart supports hover inspection, box zoom, pan, autoscale, and image export. Values are never combined across incompatible units.

In [8]:
chart_series = []
for catalog_entry in sorted(series_catalog, key=lambda row: row["parameter"]):
    points = sorted(
        points_by_series[catalog_entry["external_id"]],
        key=lambda row: row["timestamp"],
    )
    chart_series.append(
        {
            "external_id": catalog_entry["external_id"],
            "parameter": catalog_entry["parameter"],
            "unit": catalog_entry["unit"],
            "task_id": catalog_entry["task_id"],
            "realization": catalog_entry["realization"],
            "object": catalog_entry["object"],
            "x": [row["timestamp"].isoformat() for row in points],
            "y": [row["value"] for row in points],
        }
    )

plot_payload = json.dumps(chart_series).replace("</", "<\\/")
explorer_html = f"""
<div id="autogration-run-explorer" style="font-family:Segoe UI,sans-serif;border:1px solid #d1d5db;padding:14px;background:#fff">
  <label for="autogration-series-select" style="display:block;font-size:12px;font-weight:600;margin-bottom:5px">Result series</label>
  <select id="autogration-series-select" style="min-width:300px;padding:7px;border:1px solid #9ca3af"></select>
  <div id="autogration-run-chart" style="height:500px;margin-top:10px"></div>
  <div id="autogration-plot-status" style="font-size:12px;color:#6b7280">Loading Plotly.js...</div>
</div>
<script>
(() => {{
  const series = {plot_payload};
  const select = document.getElementById("autogration-series-select");
  const status = document.getElementById("autogration-plot-status");
  for (const item of series) {{
    const option = document.createElement("option");
    option.value = item.external_id;
    option.textContent = `${{item.parameter}} — ${{item.object}} (${{item.unit}})`;
    if (item.parameter === "OilRate") option.selected = true;
    select.appendChild(option);
  }}
  const render = () => {{
    const item = series.find(candidate => candidate.external_id === select.value) || series[0];
    Plotly.react(
      "autogration-run-chart",
      [{{
        x: item.x,
        y: item.y,
        type: "scatter",
        mode: "lines+markers",
        line: {{color: "#006f62", width: 2}},
        marker: {{color: "#d9482b", size: 5}},
        customdata: item.x.map(() => [item.task_id, item.realization, item.external_id]),
        hovertemplate: "%{{x|%Y-%m-%d}}<br>%{{y:,.3f}} " + item.unit +
          "<br>task: %{{customdata[0]}}<br>realization: %{{customdata[1]}}" +
          "<br>series: %{{customdata[2]}}<extra></extra>"
      }}],
      {{
        title: {{text: item.parameter + " — " + item.object, font: {{size: 18}}}},
        xaxis: {{title: "Simulation date", rangeslider: {{visible: true}}}},
        yaxis: {{title: item.unit, automargin: true}},
        margin: {{t: 55, r: 25, b: 80, l: 85}},
        hovermode: "x unified",
        paper_bgcolor: "#ffffff",
        plot_bgcolor: "#f7f7f5"
      }},
      {{responsive: true, displaylogo: false}}
    );
    status.textContent = "Task " + item.task_id + " · realization " + item.realization + " · " + item.external_id;
  }};
  select.addEventListener("change", render);
  const start = () => {{
    if (window.Plotly) {{ render(); return; }}
    const script = document.createElement("script");
    script.src = "https://cdn.plot.ly/plotly-2.35.2.min.js";
    script.onload = render;
    script.onerror = () => {{ status.textContent = "Plotly.js could not be loaded. Check network access."; }};
    document.head.appendChild(script);
  }};
  start();
}})();
</script>
"""
display(HTML(explorer_html))

## Validation

These assertions are deliberately tied to the selected export. They prevent an empty, malformed, or mislabeled result from producing a persuasive chart.

In [9]:
assert source_kind in {"live_api", "checked_in_export_fixture"}
assert len(series_catalog) == len(exported_series) > 0
assert len(observations) == sum(series_counts.values()) > 0
assert missing_values == 0
assert duplicate_points == 0
assert missing_identity == 0
assert missing_units == 0
assert {"OilRate", "WCT"}.issubset(inventory["parameters"])
assert all(len(item["x"]) == len(item["y"]) > 0 for item in chart_series)

print(
    f"Validated {len(series_catalog)} series and {len(observations)} datapoints "
    f"from {source_kind}."
)

Validated 18 series and 882 datapoints from checked_in_export_fixture.


### Q&A

- **Was a simulated run fetched?** Yes. The notebook loaded and validated Autogration's checked-in CDF-shaped RESOLVE export because the live API timed out. It did not claim a live CDF fetch.
- **Can engineers explore it interactively?** Yes. The selector exposes all 18 result series with units, provenance, hover details, zoom, pan, and range selection.

### Data Analysis Key Findings

- The run contains 18 series and 882 monthly datapoints from January 2025 through January 2029, with complete identity and units and no null or duplicate series/timestamp values.
- Oil rate falls from 4,999.9 to 126.663 Sm3/day, water cut reaches 90.799%, and cumulative oil reaches 2.046 MSm3.

### Insights or Next Steps

- Restore PostgreSQL/Docker or obtain approved CDF read access, then rerun the acquisition cell against a current completed task and compare its exported schema with this fixture.
- Use the accompanying Goose/CDF assessment to implement a governed read-only run catalog before adding historical classification or CDF links.